> 在线讲义：[第 50 章 · 解码与采样：温度、top-k / top-p、beam search、投机解码、约束解码](https://wikieden.github.io/math-and-eval-stats/lessons/50-decoding.html) · 本书《数学与实验统计》

# 第 50 章 · 解码与采样：温度、top-k / top-p、beam search、投机解码、约束解码

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/50-decoding.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 温度 | z/T | `softmax(LOGITS / t)` | T = 0.5：0.656 … |
| 第二步 top-k / top-p | 截断 | `top_k(p, 2)`、`top_p(p, 0.8)` | 留 2 个 / 3 个 |
| 第三步 beam | 整句概率 | `greedy_vs_best()` | 0.24 vs 0.36 |
| 第四步 投机解码 | 解析输出分布 | `speculative_output(p, q)` | 等于 p，α = 0.7738 |
| 第四步 期望 token | 几何级数 | `expected_tokens(alpha, 4)` | 3.19 |

例子：6 个候选 token 的 logits [2, 1.5, 1, 0, −1, −2]；温度、top-k、top-p；一个两步的小树上贪心不等于最优；
投机解码的接受率、每步期望 token 数，以及“接受—拒绝后重采”的输出分布恰好等于目标分布（解析计算）；约束解码。
里程碑 ME-M30 的“实现投机采样并用 10 万次采样验证输出分布不变”留给自己做。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np
LOGITS = np.array([2.0, 1.5, 1.0, 0.0, -1.0, -2.0])

**函数 `softmax`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def softmax(z: np.ndarray) -> np.ndarray:
    e = np.exp(z - z.max())
    return e / e.sum()

**函数 `top_k`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def top_k(p: np.ndarray, k: int) -> np.ndarray:
    q = np.zeros_like(p)
    idx = np.argsort(-p)[:k]
    q[idx] = p[idx]
    return q / q.sum()

**函数 `top_p`**：核采样：按概率从大到小累加，保留累计概率刚好 ≥ thr 的最小集合。

In [ ]:
def top_p(p: np.ndarray, thr: float) -> np.ndarray:
    """核采样：按概率从大到小累加，保留累计概率刚好 ≥ thr 的最小集合。"""
    order = np.argsort(-p)
    keep = order[: int(np.searchsorted(np.cumsum(p[order]), thr) + 1)]
    q = np.zeros_like(p)
    q[keep] = p[keep]
    return q / q.sum()

In [ ]:
TREE = {"A": (0.6, {"x": 0.4, "y": 0.3, "z": 0.3}), "B": (0.4, {"x": 0.9, "y": 0.1})}

**函数 `greedy_vs_best`**：两步：第一步选 A（0.6）或 B（0.4），第二步按条件概率。贪心每步取最大；beam 宽 2 保留两个前缀再比整句概率。

In [ ]:
def greedy_vs_best() -> tuple[tuple[str, float], tuple[str, float]]:
    """两步：第一步选 A（0.6）或 B（0.4），第二步按条件概率。贪心每步取最大；beam 宽 2 保留两个前缀再比整句概率。"""
    a = max(TREE, key=lambda k: TREE[k][0])
    g2 = max(TREE[a][1], key=TREE[a][1].get)
    greedy = (a + g2, TREE[a][0] * TREE[a][1][g2])
    cands = [(k1 + k2, p1 * p2) for k1, (p1, nxt) in TREE.items() for k2, p2 in nxt.items()]
    return greedy, max(cands, key=lambda c: c[1])

**函数 `speculative_output`**：草稿模型按 q 提议 x，以 min(1, p(x)/q(x)) 接受；拒绝时从 norm(max(0, p − q)) 重采。

In [ ]:
def speculative_output(p: np.ndarray, q: np.ndarray) -> tuple[np.ndarray, float]:
    """草稿模型按 q 提议 x，以 min(1, p(x)/q(x)) 接受；拒绝时从 norm(max(0, p − q)) 重采。
    输出分布 = q·min(1, p/q) + (1 − α)·norm(max(0, p − q))，α = Σ min(p, q)。返回 (输出分布, α)。"""
    acc = np.minimum(p, q)
    alpha = acc.sum()
    resid = np.maximum(p - q, 0)
    return acc + (1 - alpha) * resid / resid.sum(), float(alpha)

**函数 `expected_tokens`**：一次提议 γ 个草稿 token、每个独立以概率 α 被接受：每次大模型前向期望产出 (1 − α^{γ+1})/(1 − α) 个 token（Leviathan 2022）。

In [ ]:
def expected_tokens(alpha: float, gamma: int) -> float:
    """一次提议 γ 个草稿 token、每个独立以概率 α 被接受：每次大模型前向期望产出 (1 − α^{γ+1})/(1 − α) 个 token（Leviathan 2022）。"""
    return (1 - alpha ** (gamma + 1)) / (1 - alpha)

**函数 `constrained`**：约束解码：不允许的 token 概率置 0（logit 填 −∞）再归一化。

In [ ]:
def constrained(p: np.ndarray, allowed: list[int]) -> np.ndarray:
    """约束解码：不允许的 token 概率置 0（logit 填 −∞）再归一化。"""
    q = np.zeros_like(p)
    q[allowed] = p[allowed]
    return q / q.sum()

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
p = softmax(LOGITS)

In [ ]:
print("\n第一步 温度")
for t in (0.5, 1.0, 2.0):
    print(f"  T = {t}：{np.round(softmax(LOGITS / t), 3).tolist()}")

In [ ]:
print("\n第二步 top-k 与 top-p")
print(f"  原分布 {np.round(p, 3).tolist()}")
print(f"  top-k = 2：{np.round(top_k(p, 2), 3).tolist()}")
print(f"  top-p = 0.8：{np.round(top_p(p, 0.8), 3).tolist()}")

In [ ]:
print("\n第三步 贪心不一定得到概率最大的整句")
(gs, gp), (bs, bp) = greedy_vs_best()
print(f"  贪心：{gs}，整句概率 {gp:.2f}；beam 宽 2 找到：{bs}，整句概率 {bp:.2f}")

In [ ]:
print("\n第四步 投机解码：输出分布不变")
q = softmax(np.array([1.5, 2.0, 0.5, 0.0, -1.0, -1.0]))
out, alpha = speculative_output(p, q)
print(f"  目标 p {np.round(p, 4).tolist()}")
print(f"  草稿 q {np.round(q, 4).tolist()}")
print(f"  接受—重采后的分布 {np.round(out, 4).tolist()}（和 p 的最大差 {np.abs(out - p).max():.1e}），接受率 α = {alpha:.4f}")
for g in (1, 4, 8):
    print(f"  一次提议 γ = {g} 个：每次大模型前向期望得到 {expected_tokens(alpha, g):.2f} 个 token")

In [ ]:
print("\n第五步 约束解码：只允许 token 2、3、4")
print(f"  {np.round(constrained(p, [2, 3, 4]), 3).tolist()}")

## 3. 代码里才看得到的

1. **投机解码的分布是解析算的**：直接按公式把接受部分和重采部分加起来，没有做采样；用 10 万次采样实测输出分布不变，留给自己做（里程碑 ME-M30）。
2. **期望 token 数假设每个草稿独立以 α 接受**：实际接受率和位置、上下文有关，前面的草稿通常更容易被接受。
3. **top-p 用 `searchsorted`**：找累计和第一次 ≥ p 的位置，再 +1 得到保留的个数。

## 4. 自己改一改

1. 把 q 改得离 p 更远（比如均匀分布），α 和 γ = 4 的期望 token 数变成多少？
2. 给 `greedy_vs_best` 的树再加一层，构造一个 beam 宽 2 也找不到最优整句的例子。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 温度除 logits，小尖大平零贪心；
> top-k 留个数，top-p 留够概率、随分布伸缩；
> 每步最大不等于整句最大，beam 保留多个前缀；
> 投机解码 min(1, p/q) 接受、(p−q)⁺ 重采，分布不变；
> 期望 token 数 (1 − α^{γ+1})/(1 − α)；约束解码非法置零。